# QuantiPhy baseline on Kaggle (Track B, open-weight)

Settings: **Internet on**, accelerator **GPU T4 x2**.

Runs several VLM configurations on the labelled validation split (158 questions, 24 videos) and prints one comparison table. Each run writes `outputs/<name>.csv` (predictions, resumable) and `outputs/<name>.json` (MRA by category and by source).

If an earlier version of this notebook ran `pip install -U -r requirements.txt`, the session's numpy is broken: use **Run > Factory reset** (or stop and restart the session) before running again; restarting only the kernel is not enough.


In [ ]:
%cd /kaggle/working
!rm -rf quantiphy-agentic && git clone --depth 1 -b improve-baseline https://github.com/OxyzGiaHuy/quantiphy.git quantiphy-agentic
%cd /kaggle/working/quantiphy-agentic
# Only add what the Kaggle image lacks; never reinstall numpy/pandas/opencv/torch (breaks binaries).
!pip install -q -r requirements-kaggle.txt
!python -c "import numpy, pandas, torch, transformers, cv2; print('numpy', numpy.__version__, '| pandas', pandas.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| opencv', cv2.__version__)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id='PaulineLi/QuantiPhy-validation', repo_type='dataset',
                  local_dir='/kaggle/working/quantiphy-validation',
                  allow_patterns=['validation_dataset.csv', 'validation_videos/*.mp4'])
CSV = '/kaggle/working/quantiphy-validation/validation_dataset.csv'
VIDEOS = '/kaggle/working/quantiphy-validation/validation_videos'

## Configurations
Start with `LIMIT = 5` to check memory and speed, then set `LIMIT = 0` for all 158 rows.
Comment out runs you do not need. `qwen3vl8b` in fp16 needs both T4s; on a single T4 add `--load-4bit`.

In [ ]:
LIMIT = 5
RUNS = {
    # name: extra CLI flags
    'qwen25vl3b_direct':  '--model Qwen/Qwen2.5-VL-3B-Instruct --frames 12',
    'qwen3vl8b_direct':   '--model Qwen/Qwen3-VL-8B-Instruct --frames 12',
    'qwen3vl8b_measure':  '--model Qwen/Qwen3-VL-8B-Instruct --frames 12 --prompt-style measure',
}

In [ ]:
import subprocess
from pathlib import Path
Path('outputs').mkdir(exist_ok=True)
for k, (name, flags) in enumerate(RUNS.items()):
    print(f'===== {name} =====', flush=True)
    cmd = (f'python run_local_baseline.py --csv {CSV} --video-dir {VIDEOS} '
           f'--output outputs/{name}.csv --limit {LIMIT} {flags}')
    rc = subprocess.run(cmd, shell=True).returncode  # separate process frees GPU memory between runs
    if not Path(f'outputs/{name}.csv').is_file():
        print(f'{name}: no predictions written (exit code {rc}).')
        if k == 0:
            print('First run failed before predicting: fix the environment before trying the others.')
            break
        continue
    subprocess.run(f'python score_validation.py --predictions outputs/{name}.csv --output outputs/{name}.json', shell=True)

In [ ]:
import json, pandas as pd
from pathlib import Path
rows = {p.stem: json.loads(p.read_text()) for p in sorted(Path('outputs').glob('*.json')) if not p.name.endswith('.config.json')}
cols = ['rows','invalid_predictions','mra_macro','mra_S2','mra_D2','mra_S3','mra_D3','seconds_per_row']
if rows:
    display(pd.DataFrame(rows).T.reindex(columns=cols).style.format(precision=3))
else:
    print('No scored runs yet.')

In [ ]:
# Inspect the worst answers of one run
name = 'qwen3vl8b_direct'
df = pd.read_csv(f'outputs/{name}.csv')
df['rel_err'] = (df.parsed_value - df.ground_truth_posterior).abs() / df.ground_truth_posterior
display(df.sort_values('rel_err', ascending=False)[['video_id','question','ground_truth_prior','ground_truth_posterior','parsed_value','raw_response']].head(15))